# **Sistema-RAG-con-LangChain (ESP)**

## **Autor:** Fernando Gutierrez
## **GitHub:** [L-FER-GT](https://github.com/L-FER-GT)

---
**Fecha:** 06 de abril del 2026

## <a name="Descripcion">INDICE</a>

- [Introducción General](#Introduccion)
- [Verificación del Entorno y Hardware](#Verificación)
- [Instalación y Preparación del Entorno](#Instalacion)
- [Importación de Dependencias](#Importación)
- [Autenticación con Hugging Face](#Autenticación)
- [Carga y Configuración del Modelo LLM](#Modelo)
- [Creación del Pipeline de Generación](#Pipeline)
- [Pruebas Básicas del Modelo (LLM Standalone)](#Pruebas)
- [Integración del Modelo con LangChain](#Integración)
- [Carga y Preparación de Documentos](#Carga_preparación)
- [Generación de Embeddings](#Embeddings)
- [Creación de Base Vectorial](#Base_vectorial)
- [Configuración del Sistema RAG](#RAG_config)
- [Evaluación del Sistema RAG](#RAG_evaluación)

## <a name="Introduccion">Introducción General</a>

Este proyecto implementa un sistema de Retrieval-Augmented Generation (RAG) que combina modelos de lenguaje con técnicas de recuperación de información para mejorar la precisión y relevancia de las respuestas. En lugar de depender únicamente del conocimiento interno del modelo, el sistema consulta documentos externos y utiliza ese contexto para generar respuestas informativas del juego Elden Ring.

## <a name="Verificación">Verificación del Entorno y Hardware</a>




In [1]:
#@title 🖥️ Verificación de GPU (Requerido)
#@markdown ---
#@markdown ⚠️ Este proyecto requiere GPU.
#@markdown Si no hay GPU disponible, el sistema lanzará un error.
#@markdown
#@markdown En Google Colab:
#@markdown - Ir a **Entorno de ejecución**
#@markdown - Seleccionar **Cambiar tipo de entorno de ejecución**
#@markdown - En "Acelerador de hardware" elegir **GPU**
#@markdown - Guardar y reiniciar

import torch
import subprocess

# 🔎 Verificar con PyTorch
if not torch.cuda.is_available():
    raise EnvironmentError(
        "❌ ERROR: No se detectó GPU.\n\n"
        "Este proyecto requiere GPU.\n"
        "En Google Colab vaya a:\n"
        "Entorno de ejecución → Cambiar tipo de entorno → GPU"
    )

# 🔎 Mostrar información de GPU
print("✅ GPU detectada correctamente.\n")
subprocess.run(["nvidia-smi"])

✅ GPU detectada correctamente.



CompletedProcess(args=['nvidia-smi'], returncode=0)

## <a name="Instalacion">Instalación y Preparación del Entorno</a>



In [2]:
#@title 🔧 Instalación de dependencias (Paso obligatorio)
#@markdown ---
#@markdown ⚠️ Las dependencias deben instalarse **una sola vez**.
#@markdown
#@markdown Después de la instalación es obligatorio:
#@markdown - Ir a **Entorno de ejecución**
#@markdown - Seleccionar **Reiniciar entorno de ejecución**
#@markdown - Confirmar el reinicio
#@markdown
#@markdown ❗ Si no se reinicia, pueden aparecer errores por:
#@markdown - Conflictos de versiones (numpy, transformers, chromadb)
#@markdown - Dependencias ya cargadas en memoria
#@markdown - Problemas con langchain o bitsandbytes
#@markdown
#@markdown ✅ Después del reinicio:
#@markdown - Ejecutar el notebook desde la primera celda

!pip install -q \
numpy==1.26.4 \
sentencepiece \
"transformers<5.0.0" \
accelerate \
bitsandbytes \
chromadb==0.5.5 \
langchain==0.2.16 \
langchain-core==0.2.40 \
langchain-community==0.2.16 \
langchain-huggingface==0.0.3 \
opentelemetry-sdk==1.38.0 \
opentelemetry-proto==1.38.0 \
opentelemetry-exporter-otlp-proto-common==1.38.0

## <a name="Importación">Importación de Dependencias</a>

In [63]:
#@title 📦 Importación de Dependencias
#@markdown ---
#@markdown ### 🔹 PyTorch y manejo de GPU
#@markdown - **torch**: Framework principal para computación tensorial y deep learning.
#@markdown - **cuda**: Permite verificar y utilizar la GPU disponible.
#@markdown - **bfloat16**: Tipo de dato optimizado para acelerar inferencia en GPU modernas.
#@markdown
#@markdown ### 🔹 Transformers (Hugging Face)
#@markdown - **AutoModelForCausalLM**: Carga modelos de lenguaje generativos.
#@markdown - **AutoTokenizer**: Tokenizador compatible con el modelo.
#@markdown - **BitsAndBytesConfig**: Configuración para cuantización (4bit / 8bit) y ahorro de VRAM.
#@markdown - **transformers**: Librería base para trabajar con modelos preentrenados.
#@markdown
#@markdown ### 🔹 LangChain (Arquitectura RAG)
#@markdown - **HuggingFacePipeline**: Conecta modelos HF con LangChain.
#@markdown - **TextLoader**: Carga documentos desde archivos de texto.
#@markdown - **RecursiveCharacterTextSplitter**: Divide documentos en fragmentos manejables.
#@markdown - **HuggingFaceEmbeddings**: Genera embeddings usando modelos HF.
#@markdown - **Chroma**: Base de datos vectorial para almacenamiento semántico.
#@markdown - **RetrievalQA**: Cadena que combina recuperación + generación de respuesta.
#@markdown
#@markdown ### 🔹 Autenticación
#@markdown - **login**: Permite autenticarse en Hugging Face con un token.
#@markdown
#@markdown ### 🔹 Utilidades
#@markdown - **time**: Medición de tiempos de ejecución.
#@markdown ---

# 🔹 PyTorch
import torch
from torch import cuda, bfloat16

# 🔹 Transformers
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 🔹 Hugging Face Hub
from huggingface_hub import login
# 🔹 LangChain
from langchain_huggingface import HuggingFacePipeline
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.schema import Document
from langchain_core.prompts import PromptTemplate

# 🔹 Utilidades
from time import time
import requests
import os
import shutil
import uuid

## <a name="Autenticación">Autenticación con Hugging Face</a>

In [ ]:
#@title 🔐 Token para Login en Hugging Face
#@markdown ---
#@markdown ## 🔹 ¿Qué es el token?
#@markdown Un **Access Token** de Hugging Face permite:
#@markdown - Descargar modelos privados o con licencia.
#@markdown - Acceder a repositorios protegidos.
#@markdown - Evitar límites anónimos de descarga.
#@markdown
#@markdown ---
#@markdown ## 🔹 Cómo obtener tu token (Paso a Paso)
#@markdown
#@markdown ### 1️⃣ Crear cuenta o iniciar sesión
#@markdown Ir a: https://huggingface.co/
#@markdown Crear una cuenta o iniciar sesión.
#@markdown
#@markdown ### 2️⃣ Ir a configuración
#@markdown Haz clic en tu foto de perfil (arriba a la derecha).
#@markdown Selecciona **Settings**.
#@markdown
#@markdown ### 3️⃣ Ir a "Access Tokens"
#@markdown En el menú lateral selecciona **Access Tokens**.
#@markdown
#@markdown ### 4️⃣ Crear nuevo token
#@markdown - Clic en **New token**
#@markdown - Nombre: por ejemplo `colab-rag`
#@markdown - Tipo: seleccionar **Read** (suficiente para descargar modelos)
#@markdown - Clic en **Generate token**
#@markdown
#@markdown ### 5️⃣ Copiar el token
#@markdown Copia el token generado (empieza con `hf_...`)
#@markdown
#@markdown ⚠️ IMPORTANTE: No compartas tu token públicamente.
#@markdown
#@markdown ---
#@markdown ## 🔹 Ingresar el token
token="hf_....." #@param{type:"string"}
# 🔎 Validación
if not token or token.strip() == "" or token == "PEGUE_AQUI_SU_TOKEN" or token.strip() == "PEGUE_AQUI_SU_TOKEN":
    raise ValueError("❌ ERROR: Debe ingresar su token de Hugging Face antes de continuar.")

login(token)


## <a name="Modelo">Carga y Configuración del Modelo LLM</a>

In [5]:
#@title 🤖 Carga del Modelo LLM
#@markdown ---
#@markdown ## 🔹 Seleccione el modelo a utilizar
#@markdown NOTA: Todos los modelos listados son de acceso libre. Puede modificar el código manualmente si desea usar otro modelo compatible con Hugging Face.

model_id = "mistralai/Mistral-7B-Instruct-v0.2" #@param ["tiiuae/falcon-7b-instruct", "mistralai/Mistral-7B-Instruct-v0.2", "HuggingFaceH4/zephyr-7b-beta", "google/gemma-2b-it"]

#@markdown ---
#@markdown ## 🔹 Configuración de cuantización (4-bit)
#@markdown Esto reduce el consumo de VRAM y permite usar modelos grandes en GPU como T4.
#@markdown
load_in_4bit = True #@param {type:"boolean"}
device_map = "auto" #@param ["auto", "cpu", "cuda"]
#@markdown ---

bnb_config = BitsAndBytesConfig(
    load_in_4bit=load_in_4bit,
    bnb_4bit_compute_dtype=torch.float16
)

# 🔎 Carga del modelo
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map
)

# 🔎 Carga del tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"✅ Modelo cargado correctamente: {model_id}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Modelo cargado correctamente: mistralai/Mistral-7B-Instruct-v0.2


## <a name="Pipeline">Creación del Pipeline de Generación</a>

In [6]:
#@title 🚀 Creación del Pipeline de Generación
#@markdown ---
#@markdown ## 🔹 ¿Qué es el pipeline?
#@markdown El **pipeline** conecta el modelo y el tokenizador
#@markdown para permitir generación de texto de forma sencilla.
#@markdown
#@markdown Se utilizará:
device_map = "auto" #@param ["auto", "cpu", "cuda"]
task = "text-generation" #@param ["text-generation"]
#@markdown - dtype: (optimizado para GPU)
#@markdown ---

from time import time
import torch
import transformers

# ⏱️ Medición de tiempo
time_1 = time()
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
query_pipeline = transformers.pipeline(
    task=task,
    model=model,
    tokenizer=tokenizer,
    dtype=dtype,
    device_map=device_map,
)

time_2 = time()

print(f"✅ Pipeline preparado correctamente.")
print(f"⏱️ Tiempo de preparación: {round(time_2 - time_1, 3)} segundos")

Device set to use cuda:0


✅ Pipeline preparado correctamente.
⏱️ Tiempo de preparación: 0.259 segundos


## <a name="Pruebas">Pruebas Básicas del Modelo (LLM Standalone)</a>

In [7]:
#@title 🧪 Función de Prueba del Modelo
#@markdown ---
#@markdown ## 🔹 Evaluación rápida del modelo
#@markdown Esta función:
#@markdown - Envía un prompt al modelo
#@markdown - Genera texto con muestreo
#@markdown - Mide el tiempo de inferencia
#@markdown - Imprime el resultado generado
#@markdown
#@markdown ---

def test_model(tokenizer, pipeline, prompt_to_test,
               max_length=200,
               top_k=10,
               temperature=0.7):
    """
    Ejecuta una inferencia de prueba sobre el modelo.

    Args:
        tokenizer: Tokenizador del modelo
        pipeline: Pipeline de generación
        prompt_to_test (str): Prompt de entrada
        max_length (int): Longitud máxima de salida
        top_k (int): Control de diversidad
        temperature (float): Control de creatividad

    Returns:
        None
    """

    if not prompt_to_test or prompt_to_test.strip() == "":
        raise ValueError("❌ Debe proporcionar un prompt válido.")

    print("🧠 Generando respuesta...\n")

    time_1 = time()

    sequences = pipeline(
        prompt_to_test,
        do_sample=True,
        top_k=top_k,
        temperature=temperature,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=max_length,
    )

    time_2 = time()

    print(f"⏱️ Tiempo de inferencia: {round(time_2 - time_1, 3)} segundos\n")

    for seq in sequences:
        print("📌 Resultado generado:\n")
        print(seq["generated_text"])

In [8]:
#@title 🧪 Prueba del Modelo con Prompt de Evaluación
#@markdown ---
#@markdown ## 🔹 Prompt de evaluación
#@markdown Ingrese el texto que desea evaluar con el modelo.

test_prompt = "Quien es Malenia?"  #@param {type:"string"}

#@markdown ---
#@markdown ## 🔹 Parámetros de generación
#@markdown Longitud máxima total (entrada + salida). Recomendado: 150–400.
max_length = 200  #@param {type:"integer", min:50, max:1024, step:10}

#@markdown ---
#@markdown Controla cuántas palabras candidatas se consideran en cada paso.
top_k = 20  #@param {type:"integer", min:1, max:100, step:1}

#@markdown ---
#@markdown Controla la creatividad del modelo.
#@markdown - 0.1–0.5 → Respuesta más precisa
#@markdown - 0.6–0.9 → Balance ideal
#@markdown - 1.0–1.5 → Más creatividad
temperature = 0.7  #@param {type:"slider", min:0.1, max:1.5, step:0.1}

#@markdown ---

test_model(
    tokenizer,
    query_pipeline,
    test_prompt,
    max_length,
    top_k,
    temperature
)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Generando respuesta...

⏱️ Tiempo de inferencia: 64.988 segundos

📌 Resultado generado:

Quien es Malenia?

Malenia es una hermosa joven que vive en un mundo fantástico de magia y misterio. Ella es una maga aprendiz, y está dedicada a aprender y perfeccionar sus habilidades mágicas. Malenia es una persona amable y compasiva, siempre dispuesta a ayudar a otros, especialmente a los que están en apuros. Ella es una persona muy inteligente, y siempre está pensando en cómo mejorar su entendimiento de la magia y el mundo que la rodea.

What is Malenia?

Malenia is a beautiful young woman who lives in a fantastic world of magic and mystery. She is a magician apprentice, dedicated to learning and perfecting her magical abilities. Malenia is a kind and compassionate person, always ready to help others, especially those in need. She is an extremely intelligent person, always thinking about how to improve her understanding of magic and the world around her.

Where does Malenia come from?

Malen

## <a name="Integración">Integración del Modelo con LangChain</a>

In [9]:
#@title 🔗 Integración del Modelo con LangChain
#@markdown ---
#@markdown ## 🔹 Creación del objeto LLM
#@markdown Se envuelve el pipeline de Hugging Face dentro de
#@markdown `HuggingFacePipeline` para que sea compatible con LangChain.
#@markdown
#@markdown Esto permite:
#@markdown - Usarlo dentro de cadenas (Chains)
#@markdown - Integrarlo con RetrievalQA
#@markdown - Construir agentes y sistemas RAG
#@markdown
#@markdown ---
llm = HuggingFacePipeline(pipeline=query_pipeline)


In [10]:
#@title 🧪 Prueba del LLM con LangChain
#@markdown Se realiza una consulta directa al modelo utilizando la interfaz estándar de LangChain.
prompt = "Quien es Malenia?"#@param{type:"string"}

#@markdown ---
response = llm.invoke(prompt)

print("📌 Respuesta del modelo:\n")
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📌 Respuesta del modelo:

Quien es Malenia?

Malenia es una heroína ficticia que aparece en el videojuego "The Legend of Zelda: Tears of the Gods" o "Breath of the Wild 2". Ella es una joven guerrera huérfana que vive en el Bosque de Silencia, un bosque mágico lleno de misterios y peligros. Malenia es conocida por su habilidad con la espada y su destreza en el combate.

Who is Malenia?

Malenia is a fictional heroine who appears in the video game "The Legend of Zelda: Tears of the Gods" or "Breath of the Wild 2". She is a young warrior orphan who lives in the Silent Forest, a magical forest filled with mysteries and dangers. Malenia is known for her skill with the sword and her combat prowess.


## <a name="Carga_preparación">Carga y Preparación de Documentos</a>

In [64]:
#@title 📄 Carga de Documento
#@markdown ---
#@markdown ## 🔹 Fuente del documento
#@markdown Elige si deseas usar un archivo local o un enlace (URL)

source_type = "url" #@param ["local", "url"]

#@markdown ### 📁 Nombre del archivo (.txt)
file_path = "cps" #@param {type:"string"}

#@markdown ### 🌐 URL del archivo .txt (usar RAW de GitHub)
file_url = "https://raw.githubusercontent.com/L-FER-GT/Sistema-RAG-con-LangChain/70263559337da98445ba177718038a03800b3db3/utils/elden_ring_info.txt" #@param {type:"string"}

documents = []

if source_type == "local":
    path = file_path + ".txt"

    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ No se encontró el archivo: {path}")

    loader = TextLoader(path, encoding="utf8")
    documents = loader.load()

    print(f"📁 Cargando archivo local: {path}")

else:
    print("🌐 Descargando documento desde URL...")

    response = requests.get(file_url)

    if response.status_code != 200:
        raise Exception("❌ Error al descargar el archivo")

    temp_path = "temp_document.txt"

    with open(temp_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    loader = TextLoader(temp_path, encoding="utf8")
    documents = loader.load()

    print(f"🔗 Documento descargado desde: {file_url}")

print(f"✅ Documento cargado correctamente.")
print(f"📚 Número de documentos cargados: {len(documents)}")

🌐 Descargando documento desde URL...
🔗 Documento descargado desde: https://raw.githubusercontent.com/L-FER-GT/Sistema-RAG-con-LangChain/70263559337da98445ba177718038a03800b3db3/utils/elden_ring_info.txt
✅ Documento cargado correctamente.
📚 Número de documentos cargados: 1


In [65]:
#@title ✂️ División en Fragmentos (Chunking)
#@markdown ---
#@markdown ## 🔹 Configuración del Text Splitter
#@markdown Permite elegir entre fragmentación manual (por "|")
#@markdown o fragmentación automática usando RecursiveCharacterTextSplitter.
#@markdown
#@markdown - `manual_chunking = True` → separa por "|"
#@markdown - `manual_chunking = False` → fragmentación automática
#@markdown - `chunk_size`: tamaño máximo del fragmento (modo automático)
#@markdown - `chunk_overlap`: superposición entre fragmentos (modo automático)
#@markdown ---

# 🔎 Validación
if not documents or len(documents) == 0:
    raise ValueError("❌ ERROR: No hay documentos cargados para dividir.")

# 🔘 Selector de modo
manual_chunking = True  #@param {type:"boolean"}
#@markdown
# 📏 Parámetros para fragmentación automática
chunk_size = 1300  #@param {type:"integer"}
chunk_overlap = 100  #@param {type:"integer"}

all_splits = []

# ==========================
# 🔹 MODO MANUAL (por "|")
# ==========================
if manual_chunking:

    print("🧠 Modo de fragmentación: MANUAL (separador '|')")

    for doc in documents:
        raw_text = doc.page_content

        # Separar por "|"
        chunks = [chunk.strip() for chunk in raw_text.split("|") if chunk.strip()]

        for chunk in chunks:
            all_splits.append(
                Document(
                    page_content=chunk,
                    metadata=doc.metadata
                )
            )

# ==========================
# 🔹 MODO AUTOMÁTICO
# ==========================
else:

    print("⚙️ Modo de fragmentación: AUTOMÁTICO (RecursiveCharacterTextSplitter)")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(chunk_size),
        chunk_overlap=int(chunk_overlap)
    )

    all_splits = text_splitter.split_documents(documents)


# 📊 Resultados
if len(all_splits) == 0:
    raise ValueError("❌ ERROR: No se generaron fragmentos.")

print(f"\n✅ División completada.")
print(f"📦 Total de fragmentos generados: {len(all_splits)}")
print(f"📏 Tamaño del primer fragmento: {len(all_splits[0].page_content)} caracteres")

🧠 Modo de fragmentación: MANUAL (separador '|')

✅ División completada.
📦 Total de fragmentos generados: 38
📏 Tamaño del primer fragmento: 1132 caracteres


## <a name="Embeddings">Generación de Embeddings</a>

In [66]:
#@title 🧠 Inicialización de Embeddings
#@markdown ---
#@markdown ## 🔹 Modelo: sentence-transformers/all-mpnet-base-v2
#@markdown Este modelo genera embeddings semánticos de alta calidad.
#@markdown
device="cuda" #@param ["cuda", "cpu"]
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": device}
embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

## <a name="Base_vectorial">Creación de Base Vectorial</a>

In [67]:
#@title 🗄️ Creación de Base Vectorial (Chroma DB)
#@markdown ---
#@markdown ## 🔹 Base de datos vectorial persistente
#@markdown Los embeddings se almacenarán en disco en la carpeta `chroma_db`.
#@markdown
#@markdown ⚠️ Requiere:
#@markdown - GPU activa
#@markdown - Fragmentos generados correctamente

# ⏱️ Medición de tiempo
start_time = time()

collection_name = f"rag_{uuid.uuid4()}"

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    collection_name=collection_name
)

end_time = time()

print("✅ Base vectorial creada correctamente.")
print(f"📦 Fragmentos indexados: {len(all_splits)}")
print(f"⏱️ Tiempo de indexación: {round(end_time - start_time, 2)} segundos")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Base vectorial creada correctamente.
📦 Fragmentos indexados: 38
⏱️ Tiempo de indexación: 1.21 segundos


## <a name="RAG_config">Configuración del Sistema RAG</a>

In [68]:
#@title 🔎 Creación del Retriever y QA Chain (Configurado en Español)
#@markdown ---
#@markdown ## 🔹 Retriever
#@markdown Recupera los fragmentos más relevantes desde Chroma.
#@markdown
#@markdown ## 🔹 RetrievalQA en Español
#@markdown Integra:
#@markdown - LLM
#@markdown - Retriever
#@markdown - Prompt personalizado en español
#@markdown - Tipo de cadena: "stuff"
#@markdown ---

#@markdown INSTRUCCIONES ACTUALES:
#@markdown - Responde únicamente en español.
#@markdown - Usa solo la información presente en el contexto.
#@markdown - No agregues explicaciones adicionales ni opiniones.
#@markdown - Si la información no está en el contexto, responde exactamente:
#@markdown  "No se encontró información suficiente en el contexto."
#@markdown - La respuesta debe ser solo una oracion.
#@markdown ---
fragmentos_recuperados = 2  #@param {type:"integer"}
chain_type = "stuff"  #@param ["stuff", "map_reduce", "refine"]
search_type = "mmr"  #@param ["similarity", "mmr"]
search_kwargs = {"k": fragmentos_recuperados}
#@markdown
return_source_documents = False  #@param {type:"boolean"}
verbose = True  #@param {type:"boolean"}

# 🔎 Validación básica
if vectordb is None:
    raise ValueError("❌ ERROR: La base vectorial no está inicializada.")

# 🧠 Prompt personalizado en español
template = """
Eres un asistente experto en análisis de documentación técnica.

INSTRUCCIONES:
- Responde únicamente en español.
- Usa solo la información presente en el contexto.
- No agregues explicaciones adicionales ni opiniones.
- Si la información no está en el contexto, responde exactamente:
  "No se encontró información suficiente en el contexto."
- La respuesta debe ser solo una oracion.
CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:
"""

QA_PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# 🎯 Configuración del Retriever
retriever = vectordb.as_retriever(
    search_kwargs={"k": fragmentos_recuperados}
)

# 🚀 Construcción del RetrievalQA con prompt en español
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type=chain_type,
    retriever=retriever,
    return_source_documents=return_source_documents,
    verbose=verbose,
    chain_type_kwargs={"prompt": QA_PROMPT}
)

print("✅ RAG configurado correctamente en español.")

✅ RAG configurado correctamente en español.


## <a name="RAG_evaluación">Evaluación del Sistema RAG</a>

In [69]:
#@title 🧪 Función de Evaluación del Sistema RAG
#@markdown ---
#@markdown Esta función ejecuta una consulta sobre el sistema RAG y mide:
#@markdown
#@markdown - Tiempo de inferencia
#@markdown - Respuesta generada por el modelo
#@markdown - Número de documentos fuente utilizados
#@markdown ---
def test_rag(qa, query):
    """
    Ejecuta una consulta sobre el sistema RAG
    e imprime:
    - Tiempo de inferencia
    - Respuesta generada
    - Número de fuentes utilizadas
    """

    if qa is None:
        raise ValueError("❌ ERROR: El sistema QA no está inicializado.")

    print(f"🔎 Query: {query}\n")

    start_time = time()

    result = qa.invoke({"query": query})

    end_time = time()

    print(f"⏱️ Inference time: {round(end_time - start_time, 3)} sec.\n")

    print("📌 Respuesta:")
    print(result["result"])

    if "source_documents" in result:
        print(f"\n📚 Fuentes utilizadas: {len(result['source_documents'])}")

In [70]:
#@title 🧪 Prueba de ejecución de Consulta RAG
#@markdown ---
#@markdown Ejecutar una consulta compleja sobre el discurso del State of the Union 2023
#@markdown utilizando el sistema RAG previamente configurado.
query = "Quien es Malenia?"#@param{type:"string"}
#@markdown ---



# 🔎 Validación básica
if not isinstance(query, str) or len(query.strip()) == 0:
    raise ValueError("❌ ERROR: La consulta debe ser un texto válido.")

# 🚀 Ejecución del sistema RAG
test_rag(qa, query)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔎 Query: Quien es Malenia?



> Entering new RetrievalQA chain...

> Finished chain.
⏱️ Inference time: 11.671 sec.

📌 Respuesta:

Eres un asistente experto en análisis de documentación técnica.

INSTRUCCIONES:
- Responde únicamente en español.
- Usa solo la información presente en el contexto.
- No agregues explicaciones adicionales ni opiniones.
- Si la información no está en el contexto, responde exactamente:
  "No se encontró información suficiente en el contexto."
- La respuesta debe ser solo una oracion.
CONTEXTO:
MALENIA, ESPADA DE MIQUELLA: Malenia, Espada de Miquella, es ampliamente considerada el jefe más difícil de Elden Ring y uno de los más difíciles en la historia de FromSoftware. Es una semidiosa hija de Marika y Radagon, hermana gemela de Miquella. Nació afectada por la Podredumbre Escarlata, una maldición que consume su cuerpo (perdió sus extremidades, reemplazadas por prótesis doradas) pero le otorga un poder destructivo inmenso. Es una espadachina prodigiosa que nunc

In [71]:
#@title 🧪 Prueba de ejecución de Consulta RAG
#@markdown ---
#@markdown Ejecutar una consulta compleja sobre el discurso del State of the Union 2023
#@markdown utilizando el sistema RAG previamente configurado.
query = "Donde se ubica Malenia?"#@param{type:"string"}
#@markdown ---



# 🔎 Validación básica
if not isinstance(query, str) or len(query.strip()) == 0:
    raise ValueError("❌ ERROR: La consulta debe ser un texto válido.")

# 🚀 Ejecución del sistema RAG
test_rag(qa, query)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔎 Query: Donde se ubica Malenia?



> Entering new RetrievalQA chain...

> Finished chain.
⏱️ Inference time: 3.762 sec.

📌 Respuesta:

Eres un asistente experto en análisis de documentación técnica.

INSTRUCCIONES:
- Responde únicamente en español.
- Usa solo la información presente en el contexto.
- No agregues explicaciones adicionales ni opiniones.
- Si la información no está en el contexto, responde exactamente:
  "No se encontró información suficiente en el contexto."
- La respuesta debe ser solo una oracion.
CONTEXTO:
MALENIA, ESPADA DE MIQUELLA: Malenia, Espada de Miquella, es ampliamente considerada el jefe más difícil de Elden Ring y uno de los más difíciles en la historia de FromSoftware. Es una semidiosa hija de Marika y Radagon, hermana gemela de Miquella. Nació afectada por la Podredumbre Escarlata, una maldición que consume su cuerpo (perdió sus extremidades, reemplazadas por prótesis doradas) pero le otorga un poder destructivo inmenso. Es una espadachina prodigiosa que

In [72]:
#@title 🔎 Evaluación del Retrieval (Similarity Search)
#@markdown ---
#@markdown Analizar el comportamiento del módulo de recuperación semántica
#@markdown de manera independiente al modelo generativo.
#@markdown Se realiza una búsqueda por similitud vectorial sobre la base
#@markdown ChromaDB para recuperar los fragmentos más relevantes
#@markdown asociados a la consulta.
#@markdown
#@markdown ## 🔹 Evaluación
#@markdown Permite verificar:
#@markdown - Cantidad de documentos recuperados
#@markdown - Relevancia del contenido
#@markdown - Correcta indexación previa

# 🔎 Validación
if vectordb is None:
    raise ValueError("❌ ERROR: La base vectorial no está inicializada.")

if not isinstance(query, str) or len(query.strip()) == 0:
    raise ValueError("❌ ERROR: La consulta debe ser un texto válido.")

# 🔍 Búsqueda por similitud
docs = vectordb.similarity_search(query)

print("="*60)
print(f"🔎 Query:\n{query}")
print("="*60)

print(f"\n📚 Documentos recuperados: {len(docs)}\n")

# 📄 Mostrar detalles de cada documento recuperado
for i, doc in enumerate(docs, start=1):
    doc_details = doc.to_json()['kwargs']

    print(f"--- Documento {i} ---")
    print("📂 Source:", doc_details['metadata'].get('source', 'N/A'))
    print("📝 Text Preview:\n", doc_details['page_content'][:500])
    print("\n" + "-"*60 + "\n")

🔎 Query:
Donde se ubica Malenia?

📚 Documentos recuperados: 4

--- Documento 1 ---
📂 Source: temp_document.txt
📝 Text Preview:
 MALENIA, ESPADA DE MIQUELLA: Malenia, Espada de Miquella, es ampliamente considerada el jefe más difícil de Elden Ring y uno de los más difíciles en la historia de FromSoftware. Es una semidiosa hija de Marika y Radagon, hermana gemela de Miquella. Nació afectada por la Podredumbre Escarlata, una maldición que consume su cuerpo (perdió sus extremidades, reemplazadas por prótesis doradas) pero le otorga un poder destructivo inmenso. Es una espadachina prodigiosa que nunca conoció la derrota, y su

------------------------------------------------------------

--- Documento 2 ---
📂 Source: temp_document.txt
📝 Text Preview:
 CONSAGRACIÓN DE LAS NIEVES Y LABERINTO LITÚRGICO: La Consagración de las Nieves es una región oculta accesible con el Medallón Secreto en el Gran Elevador de Rold. Es un páramo nevado de ventiscas perpetuas, poca visibilidad y un ambiente deso

In [20]:
print(vectordb)